# Lab 9: Cryptography Fundamentals with Python
**Course:** FSCT 8561 -  Security Applications 

**Instructor:** Dr. Maryam R. Aliabadi

## Scenario
You have been hired as a Junior Security Engineer at CipherCorp. Your first task is to audit the company's legacy systems and migrate them to modern cryptographic standards before an upcoming security audit. 

## Learning Objectives
- Run and observe cryptographic algorithms in action.
- Understand hashing, symmetric encryption, key derivation, and secure encryption methods.
- Analyze why some cryptographic methods are secure or insecure.
- Perform a basic brute-force attack to understand the importance of salt and complexity.

## Part A — Hash Functions (One-Way Encryption)
**Goal:** Learn to generate cryptographic hashes and understand the concept of one-way encryption.
Hashing is used for integrity. If our database is stolen, we don't want a hacker to see "Password123". We want them to see a useless string.

### Task A1 — Basic Hashing 

In [1]:
import hashlib
password = input("Enter a password: ")
hash_object = hashlib.md5(password.encode())
print("md5 hash:", hash_object.hexdigest())

md5 hash: 8fe4c11451281c094a6578e6ddbf5eed


**Observations:**
- Run twice with the same password → same hash?
- Change one character → observe hash change.

**Modification:**
- Change hashing algorithm: `hashlib.sha256` → `hashlib.md5` or `hashlib.sha1`.

**Questions:**
1. What happens when you slightly change the input?
2. Compare MD5 vs SHA-256 output length.
3. Why is hashing considered “one-way”?

### Task A2 — Hashing with Salt
Salt ensures that two users with the same password (e.g., "Spring2026") do not end up with the same hash in the database, protecting against Rainbow Table attacks.

In [3]:
import uuid

def hash_password(password):
    salt = uuid.uuid4().hex
    hashed = hashlib.sha256(salt.encode() + password.encode()).hexdigest()
    return hashed + ":" + salt

def check_password(stored_password, user_input):
    hashed, salt = stored_password.split(":")
    return hashed == hashlib.sha256(salt.encode() + user_input.encode()).hexdigest()

password = input("Enter password: ")
stored = hash_password(password)
print("Stored:", stored)
verify = input("Re-enter password: ")
print("Match:", check_password(stored, verify))

NameError: name 'hashlib' is not defined

**Questions:**
1. What is the role of salt?
2. Why does the hash change every time now?

---
## Part B — Symmetric Encryption with DES
**Goal:** Understand DES encryption, input requirements, and limitations.
DES is the "antique" of the crypto world. Its 56-bit key size is no longer secure against modern computing power. We use it here to understand the mechanics of block ciphers.

In [17]:
from Crypto.Cipher import DES

key = "ouripher".encode("utf8")  # 8 bytes
message = "mes".ljust(8).encode("utf8")  # 8 bytes

cipher = DES.new(key, DES.MODE_ECB)
cipher_text = cipher.encrypt(message)
print("Encrypted:", cipher_text)

cipher = DES.new(key, DES.MODE_ECB)
plain_text = cipher.decrypt(cipher_text)
print("Decrypted:", plain_text.decode())

Encrypted: b'\x00o\xc5\xa52\x85\x16\x9f'
Decrypted: mes     


**Observations:**
- Run twice → compare outputs
- Change message slightly

**Modification:**
- Remove padding → observe error
- Fix padding with `.ljust(8)`
- Change key → observe output

**Questions:**
1. Why must DES input be exactly 8 bytes?
2. What happens if same key + same message is reused?
3. Why is this a potential security issue?

---
## Part C — AES Encryption
**Goal:** Understand AES encryption, key size, IV usage, and compare to DES.
AES is the industry standard used by governments. Unlike ECB mode used above, we use CBC (Cipher Block Chaining) with an Initialization Vector (IV) to ensure that the same message encrypted twice looks different every time.

In [24]:
from Crypto.Cipher import AES

# AES-128 requires a 16-byte key
key = "secret-key-12345".encode("utf8") 
iv = "This is an IV555".encode("utf8") 

cipher = AES.new(key, AES.MODE_CBC, iv)
# Data must be a multiple of 16 bytes
message = "This aint se".ljust(16).encode("utf8")

cipher_text = cipher.encrypt(message)
print(f"AES Encrypted: {cipher_text.hex()}")

# Decrypt using the same IV
decipher = AES.new(key, AES.MODE_CBC, iv)
plain_text = decipher.decrypt(cipher_text)
print(f"Decrypted: {plain_text.decode().strip()}")

AES Encrypted: e11a6729f83dbff2fa65eefd416b4e0d
Decrypted: This aint se


**Observations:**
- Run twice → same output?
- Change IV → observe effect

**Modification:**
- Use invalid key length → observe error
- Change message length → fix using `.ljust()`

**Questions:**
1. Why does AES require a specific key size?
2. What is the purpose of IV?
3. How is AES stronger than DES?

---
## Part D — Key Derivation (PBKDF2)
**Goal:** Learn secure key generation from passwords using PBKDF2.
Humans pick bad passwords. PBKDF2 makes it "expensive" for a hacker to guess passwords by running the hashing process thousands of times (iterations), slowing down brute-force attacks.

In [37]:
from Crypto.Protocol.KDF import PBKDF2
from Crypto import Random

password = "mypassword"
# salt = Random.new().read(16)
salt = b'fixedsalt123456'

key = PBKDF2(password, salt, dkLen=16, count=1000)
print("Key:", key)

Key: b"\x84\x93W:(]k\xb6\xb5\xd3Y'\x87\xbbJz"


**Observations:**
- Run multiple times → compare outputs

**Modification:**
- Fix salt manually: `salt = b'fixedsalt123456'`
- Change iteration count → `count = 100` to `100000`

**Questions:**
1. Why does changing salt change the key?
2. What is the role of iterations?

---
## Part E — Fernet (Simplified Secure Encryption)
**Goal:** Encrypt/decrypt safely using Fernet and observe randomness.
In modern development, we often use Fernet because it handles the IV, padding, and integrity checks automatically, preventing "rookie" mistakes in implementation.

In [76]:
from cryptography.fernet import Fernet

# Generate and save the key
key = Fernet.generate_key()
# with open("secret.key", "wb") as key_file:
#     key_file.write(key)
# with open("secret.key", "rb") as key_file:
#     key = key_file.read()

cipher = Fernet(key)

message = "Secret fsgsfdgsdfgsdfgsdfgsdfgdsfgsdfgmessage".encode()
encrypted = cipher.encrypt(message)
print("Encrypted:", encrypted)

decrypted = cipher.decrypt(encrypted)
print("Decrypted:", decrypted.decode())

Encrypted: b'gAAAAABp1H_Ts39UfhXM9NaJV6oZlGoqRm6dkHbW37ACHPIxNZgcxcPolEOSBgyWVB9FL8HXRC_57p8AQLIzUyJQjlQujJp7k3k07DsblbH8rxkL8ukUIdbvG7eQqWKb_kXKQJdg4wt9'
Decrypted: Secret fsgsfdgsdfgsdfgsdfgsdfgdsfgsdfgmessage


**Observations:**
- Run multiple times → compare outputs
- Modify key or save key to file → observe effect

**Questions:**
1. Why is Fernet safer than manual AES?
2. What happens if the key is incorrect?

---
## Part F — Broken Vault Challenge Challenge
**Goal:** Observe weaknesses in deterministic encryption and weak password hashing.
A former employee was fired for poor security practices. They left behind an encrypted "vault" and a recovery PIN that was protected with a very weak hashing method. Your mission is to recover the PIN and identify why their encryption method was a failure.

### Task F1: Brute-Forcing the Admin PIN
The employee used MD5 to hash a 4-digit PIN (e.g., "0000" to "9999") without using a salt. You have recovered the target hash: 81dc9bdb52d04dc20036dbd8313ed055.

Instructions: Complete the Python script below to "brute-force" the PIN. You must iterate through every possible 4-digit combination until you find the one that matches the target hash.

In [3]:
import hashlib

target_hash = "81dc9bdb52d04dc20036dbd8313ed055"

print("Starting Brute Force Attack...")

# --- STUDENT TODO: COMPLETE THE LOGIC BELOW ---

# 1. Create a loop that goes from 0 to 9999
# 2. Use str(i).zfill(4) to ensure numbers like 7 become "0007"
# 3. Hash the string using hashlib.md5()
# 4. Compare the result to the target_hash
# 5. Print the PIN if you find a match

# YOUR CODE HERE:
for i in range(10000):
    pin = str(i).zfill(4)
    brute = hashlib.md5(pin.encode()).hexdigest()
    if brute == target_hash:
        print(i, "is the pin")
        break


# --- END TODO ---

Starting Brute Force Attack...
1234 is the pin


### Task F2: Analyzing the ECB Leak
The employee used DES in ECB mode to encrypt two different messages using the same key. Look at the output of the code below and answer the question that follows.

In [4]:
from Crypto.Cipher import DES

key = "security".encode("utf8")
cipher = DES.new(key, DES.MODE_ECB)

# Encrypting two separate data packets
packet_1 = "DATA_REQ".encode("utf8")
packet_2 = "DATA_REQ".encode("utf8")

print(f"Packet 1 Ciphertext: {cipher.encrypt(packet_1).hex()}")
print(f"Packet 2 Ciphertext: {cipher.encrypt(packet_2).hex()}")

Packet 1 Ciphertext: 6e9501ebbf08ec18
Packet 2 Ciphertext: 6e9501ebbf08ec18


**Question:**
If an attacker intercepts these two packets and sees that the encrypted "Ciphertext" is identical, what information have they gained even if they don't have the key? Why is this a major security vulnerability for a database containing many identical values (like "Male/Female" or "Yes/No")?

---
## Deliverables
A PDF containing:
- Screenshots of code outputs for each part (A–F)
- Your completed code for the Brute-Force loop.
- Answers to all questions and observations
- Submit as PDF: `Lab9-FirstName-LastName-StudentID.pdf`

## Good Luck!